In [2]:
import sys
import os

# Use absolute paths to ensure imports work from anywhere
pipeline_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline"
eda_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline\eda"

sys.path.insert(0, eda_dir)
sys.path.insert(0, pipeline_dir)

import polars as pl
import numpy as np
import pandas as pd
from config import *
from eda_base import *
from eda_stats import *

ModuleNotFoundError: No module named 'config'

# Wage Premium Data (AKM Output - S3)
Establishment wage premiums from AKM decomposition (psi_full)

In [ ]:
# Load AKM establishment effects
import glob
akm_dir = os.path.join(OUT_DIR, 'akm')

# Load estab_fy (first year) which contains all establishment effects for first year
estab_files = sorted(glob.glob(os.path.join(akm_dir, 'estab_fy', '*.parquet')))

print(f"Found {len(estab_files)} AKM files in estab_fy/")

if estab_files:
    # Load first year only for efficiency
    estab_effects = pl.read_parquet(estab_files[0])
    
    print(f"\nShape: {estab_effects.shape}")
    print(f"\nColumns: {estab_effects.columns}")
else:
    print("No AKM files found")

In [ ]:
# Wage premium statistics (psi_full = full AKM premium, log wage units)
if 'estab_effects' in locals() and 'psi_full' in estab_effects.columns:
    premium_stats = estab_effects.select([
        pl.col('psi_full').min().alias('min_premium'),
        pl.col('psi_full').max().alias('max_premium'),
        pl.col('psi_full').mean().alias('mean_premium'),
        pl.col('psi_full').median().alias('median_premium'),
        pl.col('psi_full').std().alias('std_premium')
    ])
    
    print("Wage premium distribution (psi_full, log points):")
    for col in premium_stats.columns:
        val = premium_stats[col][0]
        print(f"  {col}: {val:.4f}")
else:
    print("psi_full column not found")

In [ ]:
# Heterogeneity: which establishments pay the highest/lowest wages?
if 'estab_effects' in locals() and 'psi_full' in estab_effects.columns:
    print("Distribution of wage premiums (percentiles):")
    percentiles = estab_effects.select([
        pl.col('psi_full').quantile(0.10).alias('p10'),
        pl.col('psi_full').quantile(0.25).alias('p25'),
        pl.col('psi_full').quantile(0.50).alias('p50'),
        pl.col('psi_full').quantile(0.75).alias('p75'),
        pl.col('psi_full').quantile(0.90).alias('p90')
    ])
    
    for col in percentiles.columns:
        val = percentiles[col][0]
        print(f"  {col}: {val:.4f}")

In [ ]:
# Wage premium categories
if 'estab_effects' in locals() and 'psi_full' in estab_effects.columns:
    mean_prem = estab_effects.select(pl.col('psi_full').mean()).to_series()[0]
    std_prem = estab_effects.select(pl.col('psi_full').std()).to_series()[0]
    
    # Categorize establishments
    high_wage = estab_effects.select(
        (pl.col('psi_full') > mean_prem + std_prem).sum()
    ).to_series()[0]
    
    low_wage = estab_effects.select(
        (pl.col('psi_full') < mean_prem - std_prem).sum()
    ).to_series()[0]
    
    medium_wage = len(estab_effects) - high_wage - low_wage
    
    print(f"Wage premium categories (relative to mean +/- 1 SD):")
    print(f"  Low-wage establishments (< mean - 1 SD): {low_wage:>6} ({low_wage/len(estab_effects)*100:>5.1f}%)")
    print(f"  Medium-wage establishments: {medium_wage:>6} ({medium_wage/len(estab_effects)*100:>5.1f}%)")
    print(f"  High-wage establishments (> mean + 1 SD): {high_wage:>6} ({high_wage/len(estab_effects)*100:>5.1f}%)")

In [ ]:
# Spatial variation in wage premiums
if 'estab_effects' in locals() and 'comt' in estab_effects.columns:
    premium_by_commune = estab_effects.group_by('comt').agg([
        pl.col('psi_full').mean().alias('avg_premium'),
        pl.len().alias('n_estabs')
    ]).sort('avg_premium', descending=True).head(15)
    
    print("Top 15 communes by average wage premium:")
    print(premium_by_commune)

In [ ]:
# Summary statistics
if 'estab_effects' in locals():
    print(f"\nAKM Establishment Premium Summary:")
    print(f"  Establishments: {len(estab_effects):,}")
    print(f"  Wage premium range: {estab_effects['psi_full'].min():.4f} to {estab_effects['psi_full'].max():.4f} (log points)")
    print(f"  Mean premium: {estab_effects['psi_full'].mean():.4f}")
    print(f"  Wage inequality (std): {estab_effects['psi_full'].std():.4f}")